In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

# 1. 패키지 설치

In [2]:
%pip install -q scikit-learn

Note: you may need to restart the kernel to use updated packages.


# 2. 환경변수 설정

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

OPENAI_LLM_MODEL = "gpt-4o-mini"
UPSTAGE_LLM_MODEL = "solar-mini4"

# OPENAI_EMBEDDING_MODEL = "text-embedding-3-large" # 차원수 3072
UPSTAGE_EMBEDDING_MODEL = "embedding-query" # 차원수 4096

PINECONE_INDEX_NAME = "better-rag-index"

# 3. category 추출
- chunk, query의 category 추출

In [4]:
def get_category():
    return {
        '납세의무': {
            '납세의무': 3, '거주자': 3, '비거주자': 3, '납세의무자': 3, 
            '원천징수': 2, '원천징수의무자': 2, '공동사업자': 2, 
            '상속인': 2, '증여자': 2, '신탁재산': 2
        },

        '세율계산': {
            '세율': 3, '소득세': 3, '과세표준': 3, '산출세액': 3, '세액': 3,
            '결정세액': 2, '세액계산': 2, '기본세율': 2, '세율적용': 2,
            '누진세율': 2, '종합소득세': 2
        },

        '근로소득': {
            '근로소득': 3, '총급여': 3, '급여': 3, '연봉': 3, '임금': 3,
            '근로소득금액': 2, '총급여액': 2, '상여': 2, '수당': 2,
            '봉급': 2, '직장인': 2
        },

        '사업소득': {
            '사업소득': 3, '총수입금액': 3, '필요경비': 3,
            '사업자': 2, '사업소득금액': 2, '결손금': 2, '이월결손금': 2,
            '주택임대소득': 2, '공동사업': 2
        },

        '이자배당': {
            '이자소득': 3, '배당소득': 3, '예금이자': 2, '채권': 2,
            '의제배당': 2, '배당세액공제': 2, '분리과세이자소득': 2,
            '분리과세배당소득': 2
        },

        '양도소득': {
            '양도소득': 3, '자산양도': 2, '부동산양도': 2, '주식양도': 2,
            '양도차익': 2, '취득가액': 2, '양도가액': 2, '양도소득금액': 2
        },

        '연금소득': {
            '연금소득': 3, '연금계좌': 3, '연금저축': 3,
            '퇴직연금': 2, '공적연금': 2, '사적연금': 2, '연금보험': 2,
            '연금수령': 2
        },

        '기타소득': {
            '기타소득': 3, '가상자산': 3, '가상자산소득': 2,
            '상금': 2, '보상금': 2, '종교인소득': 2, '원고료': 2,
            '복권': 1, '당첨금': 1, '발명보상금': 1
        },

        '공제감면': {
            '공제': 3, '소득공제': 3, '세액공제': 3,
            '기본공제': 2, '인적공제': 2, '특별공제': 2, '추가공제': 2,
            '근로소득공제': 2, '연금소득공제': 2, '퇴직소득공제': 2,
            '연금계좌세액공제': 2, '감면': 2
        },

        '비과세': {
            '비과세': 3, '비과세소득': 3, '면제': 2, '세액감면': 2,
            '소득세면제': 2, '복무급여': 1, '실업급여': 1, 
            '출산휴가급여': 1, '장학금': 1
        },

        '신고납부': {
            '신고': 3, '확정신고': 3, '과세표준확정신고': 3, 
            '납부': 3, '중간예납': 2, '가산세': 2,
            '신고기한': 2, '납부기한': 2, '납세지': 2
        },

        '과세기간': {
            '과세기간': 3, '과세연도': 3, '사업연도': 2, 
            '과세기간종료일': 2
        },

        '과세방식': {
            '종합과세': 3, '분리과세': 3, '합산과세': 2, 
            '종합소득과세표준': 2, '분리과세소득': 2, 
            '금융투자소득': 2
        },

        '장부기장': {
            '장부': 3, '복식부기': 3, '간편장부': 2, '기장': 2,
            '장부기록': 2, '증명서류': 2, '기장세액공제': 2
        }
    }

In [5]:
def categorize_content(content, top_k=None):
    """
    content의 카테고리를 분류 - 점수 기반으로 모든 카테고리를 점수순으로 list를 반환(top_k만큼)
        Parameters :
        - content : 카테고리를 분류할 텍스트
        - top_k : 상위 몇개까지의 카테고리를 반환할지 정수나 None(None이면 모든 카테고리를 반환)
        Returns :
        - 카테고리 리스트(점수순) ex.{'장부기장':30, '과세방식':45}=>['과세방식','장부기장']
    """
    category_keywords = get_category()
    category_scores = {}
    
    # 각 카테고리별 점수 계산
    for category, weighted_keywords in category_keywords.items():
        score = 0
        for keyword, weight in weighted_keywords.items():
            count = content.count(keyword) # content에 keyword가 나오는 횟수
            score += count * weight
        if score>0:
            category_scores[category] = score
    sorted_categories = sorted(category_scores.items(), key=lambda item:item[1], reverse=True)#내림차순
    all_categories = [category[0] for category in sorted_categories]
    
    # 매칭되는 카테고리가 없을 경우 ["기타"]를 반환
    if not all_categories:
        all_categories = ["기타"]
    if top_k is not None: #if top_k:
        return all_categories[:top_k]
    else:
        return all_categories

In [6]:
content = """1. 원천징수하는 자가 거주자인 경우: 그 거주자의 주된 사업장 소재지. 다만, 주된 사업장 외의 사업장에서 원천징수를 하는 경우에는 그 사업장의 소재지, 사업장이 없는 경우에는 그 거주자의 주소지 또는 거소지로 한다.
2. 원천징수하는 자가 비거주자인 경우: 그 비거주자의 주된 국내사업장 소재지. 다만, 주된 국내사업장 외의 국내사업장에서 원천징수를 하는 경우에는 그 국내사업장의 소재지, 국내사업장이 없는 경우에는 그 비거주자의 거류지(居留地) 또는 체류지로 한다.
3. 원천징수하는 자가 법인인 경우: 그 법인의 본점 또는 주사무소의 소재지
장부 장부 장부 복식부기 복식부기 복식부기 복식부기 복식부기 복식부기 복식부기 복식부기 복식부기 복식부기
"""
#content = "카테고리의 아무 단어도 없네"
print(categorize_content(content))
print(categorize_content("연봉 5천만원인 직장인의 소득세는 얼마예요?"))

['장부기장', '납세의무']
['근로소득', '세율계산']


# 4. LangChain 답변 생성

In [7]:
from langchain_upstage import ChatUpstage
from langchain_upstage import UpstageEmbeddings
# from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

C:\Users\Admin\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# 임베딩 & LLM 초기화
llm = ChatUpstage(model=UPSTAGE_LLM_MODEL)
embedding = UpstageEmbeddings(model=UPSTAGE_EMBEDDING_MODEL)

# vector DB load
vectorStore = PineconeVectorStore(
    embedding=embedding,
    index_name=PINECONE_INDEX_NAME
)

In [9]:
# 키워드 사전 활용
keyword_dict  = [
    "사람을 나타내는 표현 -> 거주자",
    "직장인 -> 근로소득이 있는 거주자", 
    "월급쟁이 -> 근로소득이 있는 거주자",
    "회사원 -> 근로소득이 있는 거주자",
    "연봉 -> 종합소득",
    "월급 -> 근로소득",
    "세금 -> 소득세",
    "공제받다 -> 공제를 적용받다",
    "얼마나 내야하나 -> 세액은 얼마인가",
    "계산해줘 -> 계산하면 얼마인가"
]

prompt = ChatPromptTemplate.from_template(f"""사용자의 질문을 보고, 우리의 사전을 참고해서 사용자 질문을
변경해 주세요. 만약 변경할 필요가 없을 경우, 사용자의 질문을 변경하지 않아도 됩니다. 오직 질문만 리턴해 주세요.
사전 : {keyword_dict}
질문 : {{question}}""")

keyword_chain = prompt | llm | StrOutputParser()
query = "연봉 5000만원인 회사원의 소득세는 얼마예요?"
normalized_query = keyword_chain.invoke({'question':query})

In [10]:
# Retriver 생성
retriever = vectorStore.as_retriever(
    search_type='similarity',
    search_kwargs = {
        'k':10,
        'filter' : {'category':{'$in':categorize_content(normalized_query)}}
    }
)

documents = retriever.invoke(normalized_query)

In [11]:
categorize_content(normalized_query)

['근로소득', '납세의무']

In [12]:
for i, doc in enumerate(documents):
    chunk_id = doc.metadata.get('chunk_id')
    category = doc.metadata.get('category')
    title = doc.metadata.get('title')
    article = doc.metadata.get('article','참조 조항 없음')
    print(f'{i+1}.{chunk_id[-3:]} : {title[:50]} - {category[0]}')

1.037 : 소득세 공제 규정 요약 - 일용근로자 공제: 1일 최대 15만원 공제 2018년 개정. - - 공제감면
2.043 : 소득세 계산 특례 및 세액공제 공동사업 소득공제 및 세액공제 적용 시 특수관계인의 지출이  - 세율계산
3.049 : 연금계좌세액공제 - 종합소득자 대상, 연금계좌 납입액의 일정 비율 공제 근로소득자 기준:  - 연금소득
4.036 : 소득공제 및 세액공제 특례 - 결손금 및 이월결손금 공제: 과세기간 결손금 발생 시 먼저  - 공제감면
5.048 : 소득세법 주요 조항 요약: - 근로소득세액공제: 총급여액에 따라 차등 공제 최대 한도 명시 - 근로소득
6.009 : 종합소득세 과세표준 계산 - 종합소득금액 계산 시 이자, 배당 등 다양한 소득 합산 후 공 - 이자배당
7.038 : 소득세법 부양가족 공제 거주자 및 배우자 포함, 직계존속재혼 시 배우자 포함, 직계비속20 - 공제감면
8.024 : 퇴직소득 및 총수입금액 계산 - 퇴직소득: 제1항 각 호 소득 합계액에서 대통령령 정한 임 - 근로소득
9.010 : 분리과세 및 분리연금소득 규정 - 분리과세 기타소득: 특정 조건 하의 연금외수령소득, 복권 - 세율계산
10.053 : 소득세법 공제 및 감면 규정 요약 - 공제 신청 요건: 거주자 신청 시 제1항제4항 적용. - 공제감면


## TFidVectorizer
- title + page_content

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def rerank_by_title(query:str, documents:list, top_k:int=3) -> list:
    '''
    sklearn의 TfidVectorizer를 사용한 documents 재정렬
    Args:
    - query : 사용자 질문
    - documents : retriever로 검색된 Document 객체 list
    - top_k : 재정렬 후 반환할 문서 개수
    Returns:
    - title 유사도 기준으로 재정렬된 Document list
    '''
    # Document 객체에서 title 추출
    titles = [doc.metadata.get('title')+' ' for doc in documents]
    
    # Tfidfvectorizer 객체
    vectorizer = TfidfVectorizer(
        analyzer='char',      # 한국어의 경우 'char' 단위 사용
        ngram_range=(2, 3)    # 단어 기준 2~3자 (튜플 형태로 전달)
    )
    
    # query, titles 벡터화
    all_texts = [query] + titles
    tfidf_matrics = vectorizer.fit_transform(all_texts)
    query_vector = tfidf_matrics[0:1] # 0번째 : 2차원 query vector
    title_vector = tfidf_matrics[1:] # 1~10번째 : 2차원 title vector
    similarities = cosine_similarity(query_vector, title_vector)[0]
    score_docs = list(zip(similarities, documents))
    
    # 유사도 기준으로 내림차순
    score_docs.sort(key=lambda x : x[0], reverse=True)
    
    # Documents 추출
    rerank_docs = [score_doc[1] for score_doc in score_docs]
    if top_k:
        return rerank_docs[:top_k]
    return rerank_docs

In [14]:
docs = rerank_by_title(normalized_query,documents, 5)
for i, doc in enumerate(docs):
    chunk_id = doc.metadata.get('chunk_id')
    category = doc.metadata.get('category')
    title = doc.metadata.get('title')
    article = doc.metadata.get('article','참조 조항 없음')
    print(f'{i+1}.{chunk_id[-3:]} : {title[:50]} - {category[0]}')

1.043 : 소득세 계산 특례 및 세액공제 공동사업 소득공제 및 세액공제 적용 시 특수관계인의 지출이  - 세율계산
2.010 : 분리과세 및 분리연금소득 규정 - 분리과세 기타소득: 특정 조건 하의 연금외수령소득, 복권 - 세율계산
3.037 : 소득세 공제 규정 요약 - 일용근로자 공제: 1일 최대 15만원 공제 2018년 개정. - - 공제감면
4.009 : 종합소득세 과세표준 계산 - 종합소득금액 계산 시 이자, 배당 등 다양한 소득 합산 후 공 - 이자배당
5.053 : 소득세법 공제 및 감면 규정 요약 - 공제 신청 요건: 거주자 신청 시 제1항제4항 적용. - 공제감면


## 참조 조항

In [15]:
docs = retriever.invoke(keyword_chain.invoke(normalized_query))

def extract_articles_from_docs(documents):
    """
    검색된 문서 documents에서 article 정보를 추출하여 리스트로 반환
    Args:
    - documents : retriever에서 반환된 Document 객체 리스트
    Returns:
        중복 제거된 조항 리스트를 문자로 바꿔서 반환 (ex. '소득세법 제4조, 제60조')
    """
    articles = []
    for doc in documents:
        # metadata에서 article 정보 추출
        article = doc.metadata.get('article', '참조 조항 없음')
        article = article.replace('「', '').replace('」', '')
        article_list = article.split(', ')
        articles.extend(article_list)
    # 중복 제거 및 정렬
    unique_article = list(set(articles))
    unique_article.sort(key=lambda x : int(x[:-1]))
    final_article = ['제'+article for article in unique_article if article!='조항없음']
    result = '소득세법 ' + ', '.join(final_article)
    return result

extract_articles_from_docs(docs)

'소득세법 제2조, 제4조, 제6조, 제12조, 제13조, 제14조, 제15조, 제16조, 제17조, 제19조, 제20조, 제21조, 제22조, 제23조, 제24조, 제25조, 제26조, 제27조, 제29조, 제31조, 제35조, 제37조, 제39조, 제41조, 제43조, 제45조, 제46조, 제47조, 제48조, 제49조, 제50조, 제51조, 제52조, 제54조, 제55조, 제59조, 제91조, 제95조, 제122조, 제127조, 제129조, 제133조, 제134조, 제146조, 제156조, 제160조'

In [16]:
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

# 5. 사용

In [17]:
# 재사용이 가능한 함수 만들기
def ask_with_reference(query:str, k:int=10, top_k:int=3):
    """
    query의 수정된 query, 답변, 참고조항을 출력하는 함수
    Args:
    - query : 질문
    - k : 검색할 문서 개수 (기본값 : 10)
    - top_k : 재정렬 후 사용할 문서 개수 (기본값 : 3)
    """
    # query 표준화
    normalized_query = keyword_chain.invoke({"question":query})

    # retriever 생성
    retriever = vectorStore.as_retriever(
        search_type = 'similarity',
        search_kwargs ={
            'k':k,
            'filter':{
                'category':{'$in':categorize_content(normalized_query)}
            }
        }
    )

    # 검색된 문서
    retrieverd_docs = retriever.invoke(normalized_query)
    
    # rerank 처리
    reranked_docs = rerank_by_title(
        query=normalized_query,
        documents=retrieverd_docs,
        top_k=top_k
    )
    
    # 참고 조항 추출
    referenced_articles = extract_articles_from_docs(reranked_docs)

    # LLM 답변 생성
    template = f"""당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 답하세요. 답을 모르면 
    모른다고 답하세요.
    질문 : {{query}}
    문맥 : {{context}}
    답변 :"""
    prompt = ChatPromptTemplate.from_template(template)
    prompt_chain = prompt | llm | StrOutputParser()
    answer = prompt_chain.invoke({
        'query':normalized_query,
        'context':format_documents(reranked_docs)
    })

    print()
    print(f'☑️ 원본 질문 : {query}')
    print(f'☑️ 질문 표준화 : {normalized_query}')
    print()
    print(f'📝 답변 : {answer}\n')
    print(f'📢 참고 조항 : {referenced_articles}')
    print('**위 답변은 AI에 의해 작성된 답변이므로 근소한 차액이 발생할 수 있습니다**')

In [18]:
ask_with_reference('연봉 5000만원인 직장인의 소득세는 얼마예요?')


☑️ 원본 질문 : 연봉 5000만원인 직장인의 소득세는 얼마예요?
☑️ 질문 표준화 : 연봉 5000만원인 근로소득이 있는 거주자의 종합소득세는 얼마인가?

📝 답변 : 질문에서 제시된 문맥만으로는 **정확한 최종 종합소득세 납부세액**을 계산할 수 없습니다.  
그 이유는 다음과 같습니다.

1. **종합소득과세표준**을 알아야 합니다.  
   - 과세표준 = 종합소득금액 − 종합소득공제  
   - 제공된 문맥에는 “연봉 5,000만원”이라는 **총급여액**만 있고, 비과세소득, 근로소득 외의 소득, 각종 공제액 등이 없습니다.

2. **근로소득공제**를 적용하려면 총급여액이 필요합니다.  
   - 문제에서 총급여액 5,000만원이라고 했으므로 근로소득공제는 계산할 수 있지만,  
   - 실제 종합소득세 계산에는 **근로소득금액 = 총급여액 − 근로소득공제**로 시작해야 합니다.  
   - 다만, 다른 인적공제·연금보험료공제·특별소득공제·그밖의 소득공제 등이 별도로 필요합니다.

3. **세율표**는 볼 수 있습니다.  
   제공된 문맥에 따르면 종합소득과세표준 5,000만원 이하 구간의 세율은 다음과 같습니다.  
   - 1,400만원 이하: 6%  
   - 1,400만원 초과 5,000만원 이하: 84만원 + (1,400만원 초과분의 15%)  
   - 따라서 과세표준이 5,000만원이면 산출세액은:  
     84만원 + (5,000만원 − 1,400만원) × 15%  
     = 84만원 + 3,600만원 × 15%  
     = 84만원 + 540만원  
     = **624만원**

   하지만 이는 **종합소득산출세액**일 뿐이며, 최종 납부할 세액은 아닙니다.

4. **세액공제**와 **납부세액 계산**은 문맥만으로 알 수 없습니다.  
   - 근로소득세액공제, 자녀세액공제, 연금저축세액공제, 특별세액공제, 표준세액공제 등이 적용될 수 있습니다.  
   - 또한 세액공제 후 감면, 가산세, 기납부세액 등이 있으

In [19]:
ask_with_reference('사업소득이 있는 경우 필요경비는 어떻게 계산하나요?')


☑️ 원본 질문 : 사업소득이 있는 경우 필요경비는 어떻게 계산하나요?
☑️ 질문 표준화 : 사업소득이 있는 경우 필요경비는 어떻게 계산하나요?

📝 답변 : 사업소득의 필요경비는 원칙적으로 **해당 과세기간의 총수입금액에 대응하는 비용으로서 일반적으로 용인되는 통상적인 것**의 합계액으로 계산합니다.

구체적인 계산 기준은 다음과 같습니다.

1. **기본 원칙(제27조 관련)**
   - 사업소득금액은 과세기간의 **총수입금액에서 필요경비를 공제한 금액**입니다.
   - 필요경비가 총수입금액을 초과하면 그 초과분은 **결손금**으로 봅니다.
   - 법령에 특별한 규정이 있는 경우 외에는, 해당 과세기간의 총수입금액에 대응하는 비용으로서 **일반적으로 용인되는 통상적인 것**의 합계액을 필요경비로 산입합니다.

2. **예외적인 필요경비 계산**
   - 제1항에 따른 경우
   - 광업권의 양도대가로 받는 금품의 필요경비 계산 등 **대통령령으로 정하는 경우**

3. **귀속 시기 관련 규정**
   - 해당 과세기간 **전의 총수입금액에 대응하는 비용**으로서 그 과세기간에 확정된 것은, 그 과세기간 전에 필요경비로 계상하지 않은 경우에만 그 과세기간의 필요경비로 봅니다.

4. **증명서류 보관 의무**
   - 사업소득금액 계산 시 필요경비를 인정받으려면, 비용의 지출에 대한 증명서류를 받아 **확정신고기간 종료일부터 5년간 보관**해야 합니다.
   - 사업과 관련하여 사업자(법인 포함)로부터 재화·용역을 공급받고 대가를 지출하는 경우에는 원칙적으로 적격 증명서류를 받아야 합니다. 다만, 대통령령으로 정하는 경우에는 예외가 있습니다.

5. **장부 비치·기록 의무**
   - 사업자는 소득금액을 계산할 수 있도록 증명서류를 갖추고, 사업 거래 사실이 객관적으로 파악되도록 장부에 기록·관리해야 합니다.
   - 업종·규모 등을 고려하여 대통령령으로 정하는 일정 규모 미만의 사업자는 **간편장부**를 기록해도 장부 비치·기록 의무를 이행한 것으